# Plot GEN files
Plot one or more GEN files side by side, e.g. to compare GENPUZZLE input and output.

In [ ]:
from pathlib import Path

import imod
import matplotlib.pyplot as plt

from genpuzzle import genpuzzle
from logging_helpers import init_logging

init_logging()

In [ ]:
data_dir = Path("tests/data")
genfile_in = data_dir / "genpuzzle_test.gen"
tolerance = 0.0001

genfiles = [genfile_in]
genfile_out = data_dir / f"out_tol{tolerance:g}.gen"
genpuzzle(genfile_in, genfile_out, tolerance)
genfiles.append(genfile_out)

In [ ]:
genfiles_to_plot = [*genfiles, data_dir / "out_imod5.gen"]

fig, axes = plt.subplots(
    1, len(genfiles_to_plot), figsize=(3.3 * len(genfiles_to_plot), 5), sharex=True, sharey=True, squeeze=False
)
for ax, genfile in zip(axes[0], genfiles_to_plot):
    gdf = imod.formats.gen.read(genfile)
    labels = gdf.get("id", gdf.index.to_series())
    gdf.plot(ax=ax, column=labels, cmap="tab10", linewidth=2)
    for line in gdf.geometry:
        (x0, y0), (x1, y1) = line.coords[0], line.coords[-1]
        ax.plot([x0, x1], [y0, y1], "ko", markersize=4)
    ax.set_title(f"{genfile.name} ({len(gdf)} lines)")
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)

test_cases = [
    ("Exact joins", (30, 5), (0, 5)),
    ("Larger gap", (21.5, 25), (0, 30)),
    ("Converging line", (20, 50), (0, 56)),
    ("Isolated line", (70, 20), (53, 43)),
    ("Floating point precision", (20, 75), (0, 83)),
    ("Three-way junction", (60, 65), (43, 87)),
    ("0.0001 gap", (20, 90), (0, 97)),
]
for label, point, text_position in test_cases:
    axes[0][0].annotate(
        label, xy=point, xytext=text_position, fontsize=8,
    )
axes[0][0].set_ylim(-15, 110)

fig.tight_layout(pad=0.5, w_pad=0.5)